# C6-pytorch — Practice p22 — Solution

In [ ]:
import numpy as np
import torch

source = np.array([[2.5, 11.0, -7.5],
                   [8.5, 17.0, -1.5]], dtype=np.float64)
before = source.copy()


def standardize_in_place(tensor):
    tensor.sub_(tensor.mean(dim=0, keepdim=True))
    return tensor


shared = torch.from_numpy(source)
standardize_in_place(shared)
mutated = bool(not np.array_equal(source, before))
shares_before_fix = bool(np.shares_memory(shared.numpy(), source))

safe_source = before.copy()
safe_tensor = torch.from_numpy(safe_source).clone()
standardize_in_place(safe_tensor)
safe_source_unchanged = bool(np.array_equal(safe_source, before))

`torch.from_numpy(source)` creates an alias edge: `shared` and `source` are two views of the same storage, so the in-place subtraction writes through to NumPy. Assigning `shared` to another Python variable would copy only the reference, not the underlying values. The repair crosses with `from_numpy` and then calls `clone()`, which gives `safe_tensor` independent torch storage before mutation. Consequently `safe_tensor` becomes the centered rows `[-3,-3,-3]` and `[3,3,3]` while `safe_source` retains the original fixture.

### Answer check

In [ ]:
expected_centered = np.array([[-3.0, -3.0, -3.0],
                              [3.0, 3.0, 3.0]])
assert mutated is True
assert shares_before_fix is True
assert np.allclose(source, expected_centered, atol=0, rtol=0)
assert not np.shares_memory(safe_tensor.numpy(), safe_source)
assert torch.allclose(
    safe_tensor,
    torch.from_numpy(expected_centered),
    atol=0,
    rtol=0,
)
assert np.allclose(safe_source, before, atol=0, rtol=0)
assert safe_source_unchanged is True